# 08 · TotalSpineSeg en Kaggle (cuaderno auxiliar)

Este cuaderno segmenta con TotalSpineSeg las series de una de las dos listas que arma el cuaderno 08. Es auxiliar, como el que se usó para SPIDER-Baseline-IIS en el cuaderno 05: Kaggle solo presta la GPU y cualquier GPU puede repetir la corrida con las mismas versiones y huellas.

**Antes de correrlo, en Kaggle:**

1. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **On** (para instalar TotalSpineSeg y bajar sus pesos).
2. *Add Input*: el dataset privado **prl-08-entradas** (lo sube la celda de entradas para Kaggle del cuaderno 08; si este cuaderno corre en la segunda cuenta, el dataset tiene que estar compartido con ella) y la competición **RSNA 2024 Lumbar Spine Degenerative Classification** (la cuenta tiene que haber aceptado sus reglas).
3. En la celda 2, `LISTA` es `"A"` en una cuenta y `"B"` en la otra, e `INICIO` es la posición de la lista desde la que se empieza (1 la primera vez).
4. *Save Version → Save & Run All (Commit)*, para que corra en segundo plano con la pestaña cerrada.

**Entre sesiones.** Kaggle corta cada sesión a las 12 horas, así que la celda de segmentación deja de tomar lotes nuevos un poco antes (`HORAS_MAXIMAS`) y al final imprime el `INICIO` de la siguiente versión. Las máscaras de cada versión quedan en un zip dentro de la salida del cuaderno, junto con su registro. Si se agrega como entrada la salida de una versión anterior, el cuaderno también salta las series que ya tengan una máscara.

Las máscaras de RSNA se quedan en la salida privada del cuaderno y después en la carpeta privada de Drive; no se publican, por los términos de RSNA.

In [1]:
# 1. Entorno aparte para TotalSpineSeg (igual que en el cuaderno 05) y lectura de DICOM
import subprocess, sys
subprocess.run("pip -q install virtualenv SimpleITK", shell=True, check=True)
subprocess.run("rm -rf /kaggle/tmp/venv_tss && virtualenv -q --system-site-packages /kaggle/tmp/venv_tss", shell=True, check=True)
subprocess.run(["/kaggle/tmp/venv_tss/bin/pip", "-q", "install", "totalspineseg==20260807", "nnunetv2==2.6.2", "kornia==0.8.2"],
               check=True)
print(subprocess.run(["/kaggle/tmp/venv_tss/bin/python", "-c",
                      "import torch; from importlib.metadata import version as v; "
                      "print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(), '| totalspineseg', v('totalspineseg'))"],
                     capture_output=True, text=True).stdout)

torch 2.11.0+cu128 | GPUs: 2 | totalspineseg 20260807



In [2]:
# 2. Parámetros y rutas
LISTA = "A"              # "A" en una cuenta, "B" en la otra
INICIO = 1               # posición de la lista desde la que se empieza esta versión
HORAS_MAXIMAS = 11.0     # se dejan de tomar lotes nuevos pasado este tiempo (Kaggle corta a las 12 h)
TAM_LOTE = 20            # series por llamada a TotalSpineSeg: el modelo se carga una vez por lote

import os, sys, time, json, shutil, hashlib, datetime, subprocess, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk

ENTRADA = Path("/kaggle/input")
TRABAJO = Path("/kaggle/working")
TMP = Path("/kaggle/tmp")
VENV_TSS = TMP / "venv_tss"
DIR_TSS_DATOS = TMP / "tss_datos"
DIR_MASCARAS = TMP / "mascaras"                     # paso intermedio; cada máscara se agrega enseguida al zip de salida
for d in (DIR_TSS_DATOS, DIR_MASCARAS):
    d.mkdir(parents=True, exist_ok=True)

def unico(patron):
    hallados = sorted(ENTRADA.rglob(patron))
    assert len(hallados) >= 1, f"No se encontró {patron} en /kaggle/input: revisa las entradas del cuaderno"
    return hallados[0]

print("Contenido de /kaggle/input (dos niveles):")
for p in sorted(q for d in ENTRADA.iterdir() if d.is_dir() for q in d.iterdir())[:30]:
    print("   ", p.relative_to(ENTRADA), "(carpeta)" if p.is_dir() else f"({p.stat().st_size / 1e6:.1f} MB)")

RUTA_LISTA = unico(f"lista_{LISTA}.csv")
RUTA_CONVERSION = unico("conversion08.py")
if not any(ENTRADA.rglob("men_*.nii*")):
    # Si Kaggle dejó la carpeta de Mendeley comprimida, se descomprime en el disco temporal
    import zipfile
    zips = [z for z in ENTRADA.rglob("*.zip") if "mendeley" in z.name.lower()]
    assert zips, "No están los NIfTI de Mendeley ni su zip: revisa en el dataset que se haya subido mendeley_nifti"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP / "mendeley_nifti")
    ENTRADA_MENDELEY = TMP / "mendeley_nifti"
else:
    ENTRADA_MENDELEY = ENTRADA
# Kaggle descomprime los .gz al procesar un dataset, así que los NIfTI pueden llegar como .nii en vez de .nii.gz
NIFTI_MENDELEY = {p.name.split(".nii")[0]: p for p in ENTRADA_MENDELEY.rglob("men_*.nii*")}
DIR_MENDELEY = next(iter(NIFTI_MENDELEY.values())).parent
print(f"NIfTI de Mendeley encontrados: {len(NIFTI_MENDELEY)} (extensión: {next(iter(NIFTI_MENDELEY.values())).name.split('.', 1)[1]})")
RSNA = [p for p in ENTRADA.rglob("train_images") if p.is_dir()]
DIR_RSNA = RSNA[0] if RSNA else None

spec = importlib.util.spec_from_file_location("conversion08", RUTA_CONVERSION)
conversion08 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(conversion08)

lista = pd.read_csv(RUTA_LISTA, dtype={"estudio": str, "serie_clave": str})
print(f"Lista {LISTA}: {len(lista)} series ({lista['conjunto'].value_counts().to_dict()})")
print("NIfTI de Mendeley en:", DIR_MENDELEY)
print("Imágenes de RSNA en:", DIR_RSNA)
assert DIR_RSNA is not None or (lista["conjunto"] == "mendeley").all(), "Falta la competición de RSNA entre las entradas"

Contenido de /kaggle/input (dos niveles):
    competitions/rsna-2024-lumbar-spine-degenerative-classification (carpeta)
    datasets/edwarfabian (carpeta)
NIfTI de Mendeley encontrados: 1030 (extensión: nii)
Lista A: 2491 series ({'rsna': 1975, 'mendeley': 516})
NIfTI de Mendeley en: /kaggle/input/datasets/edwarfabian/prl-08-entradas/mendeley_nifti
Imágenes de RSNA en: /kaggle/input/competitions/rsna-2024-lumbar-spine-degenerative-classification/train_images


In [3]:
# 3. Pesos de TotalSpineSeg de la release fijada, verificados antes de instalarlos
import requests
RELEASE_PESOS_TSS = "r20260730"
SHA256_PESOS_TSS = {
    "Dataset101_TotalSpineSeg_step1_r20260730.zip": "781f81e8c94feccd3cef17d819ba5c293b506e9b1f2b106fc5e31b9dd85887b0",
    "Dataset102_TotalSpineSeg_step2_r20260730.zip": "494c640ab8cb25475bd0fc4338c0931d37079af9dd19403879335c3908c06b46",
}
def sha256_archivo(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()

os.environ["TOTALSPINESEG_DATA"] = str(DIR_TSS_DATOS)
dir_exports = DIR_TSS_DATOS / "nnUNet" / "exports"
dir_exports.mkdir(parents=True, exist_ok=True)
for nombre, sha_esperado in SHA256_PESOS_TSS.items():
    destino = dir_exports / nombre
    if not destino.exists():
        with requests.get(f"https://github.com/neuropoly/totalspineseg/releases/download/{RELEASE_PESOS_TSS}/{nombre}",
                          stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(destino, "wb") as f:
                for trozo in r.iter_content(1 << 20):
                    f.write(trozo)
    assert sha256_archivo(destino) == sha_esperado, f"{nombre} NO coincide con la huella verificada; no continuar."
subprocess.run([str(VENV_TSS / "bin" / "totalspineseg_init"), "--quiet"], check=True, env=os.environ.copy())
print("Pesos verificados e instalados")

Pesos verificados e instalados


In [4]:
# 4. Segmentación por lotes, en las GPU disponibles a la vez
import torch
N_GPU = max(torch.cuda.device_count(), 1)
import zipfile
CSV_REG = TRABAJO / f"registro_tss_{LISTA}_desde_{INICIO:05d}.csv"
ZIP_MASCARAS = TRABAJO / f"mascaras_{LISTA}_desde_{INICIO:05d}.zip"
COLUMNAS_REG = ["lista", "orden", "conjunto", "id_serie", "estado", "geometria_ok", "huella_ok", "segundos_lote",
                "series_en_lote", "gpu", "fecha", "error"]

# Series ya segmentadas en versiones anteriores (si se agregó su salida como entrada)
hechas = set()
for reg in ENTRADA.rglob("registro_tss_*.csv"):
    r = pd.read_csv(reg, dtype=str)
    hechas |= set(r.loc[r["estado"] == "ok", "id_serie"])
pendientes = lista[(lista["orden"] >= INICIO) & ~lista["id_serie"].isin(hechas)].sort_values("orden")
print(f"GPUs: {N_GPU} | ya hechas en versiones anteriores: {len(hechas)} | pendientes desde la posición {INICIO}: {len(pendientes)}")

def preparar_entrada(r, dir_entrada):
    "Deja la serie en NIfTI en la carpeta del lote; RSNA se convierte aquí con el mismo código del cuaderno 08."
    destino = dir_entrada / f"{r.id_serie}.nii.gz"
    if r.conjunto == "mendeley":
        img = sitk.ReadImage(str(NIFTI_MENDELEY[r.id_serie]))       # puede venir como .nii: se reescribe como .nii.gz
        sitk.WriteImage(img, str(destino), useCompression=True)
        huella = conversion08.huella_volumen(img)
        return huella == r.huella_volumen, huella
    info = conversion08.convertir(DIR_RSNA / r.estudio / r.serie_clave, destino)
    return info["huella_volumen"] == r.huella_volumen, info["huella_volumen"]

def lanzar(lote, gpu):
    "Prepara un lote y lanza TotalSpineSeg en una GPU sin esperar a que termine."
    base = TMP / f"lote_gpu{gpu}"
    shutil.rmtree(base, ignore_errors=True)
    (base / "entrada").mkdir(parents=True)
    huellas = {}
    for r in lote.itertuples():
        try:
            huellas[r.id_serie] = preparar_entrada(r, base / "entrada")[0]
        except Exception as e:
            huellas[r.id_serie] = f"error de entrada: {type(e).__name__}: {e}"[:200]
    cmd = [str(VENV_TSS / "bin" / "totalspineseg"), str(base / "entrada"), str(base / "salida"),
           "--device", "cuda", "--keep-only", "step2_output", "--quiet"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "MPLBACKEND": "Agg"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=env)
    return {"lote": lote, "gpu": gpu, "base": base, "huellas": huellas, "proc": proc, "t0": time.time()}

def recoger(trabajo):
    "Espera a que termine un lote, guarda sus máscaras y anota cada serie en el registro."
    salida, error = trabajo["proc"].communicate()
    dt = time.time() - trabajo["t0"]
    filas, ok = [], 0
    for r in trabajo["lote"].itertuples():
        fila = dict.fromkeys(COLUMNAS_REG, "")
        fila.update({"lista": LISTA, "orden": r.orden, "conjunto": r.conjunto, "id_serie": r.id_serie,
                     "segundos_lote": round(dt, 1), "series_en_lote": len(trabajo["lote"]), "gpu": trabajo["gpu"],
                     "fecha": datetime.datetime.now().isoformat(timespec="seconds")})
        huella = trabajo["huellas"][r.id_serie]
        out = trabajo["base"] / "salida" / "step2_output" / f"{r.id_serie}.nii.gz"
        if isinstance(huella, str):
            fila.update({"estado": "error", "error": huella})
        elif out.exists():
            img = sitk.ReadImage(str(trabajo["base"] / "entrada" / f"{r.id_serie}.nii.gz"))
            seg = sitk.ReadImage(str(out))
            geometria_ok = conversion08.misma_geometria(img, seg)
            if geometria_ok:
                seg.CopyInformation(img)
            local = DIR_MASCARAS / f"{r.id_serie}.nii.gz"
            sitk.WriteImage(sitk.Cast(seg, sitk.sitkUInt8), str(local), useCompression=True)
            with zipfile.ZipFile(ZIP_MASCARAS, "a") as z:       # queda en la salida aunque la sesión se corte después
                z.write(local, local.name)
            local.unlink()
            fila.update({"estado": "ok", "geometria_ok": geometria_ok, "huella_ok": huella})
            ok += 1
        else:
            fila.update({"estado": "error", "error": error[-300:].replace("\n", " ")})
        filas.append(fila)
    pd.DataFrame(filas, columns=COLUMNAS_REG).to_csv(CSV_REG, mode="a", header=not CSV_REG.exists(), index=False)
    return ok, dt

lotes = [pendientes.iloc[i:i + TAM_LOTE] for i in range(0, len(pendientes), TAM_LOTE)]
inicio, siguiente, en_curso, segundos_por_lote = time.time(), 0, [], []
total_ok = 0
while siguiente < len(lotes) or en_curso:
    transcurrido = (time.time() - inicio) / 3600
    margen = (max(segundos_por_lote) if segundos_por_lote else 1800) / 3600
    libres = [g for g in range(N_GPU) if g not in {t["gpu"] for t in en_curso}]
    while libres and siguiente < len(lotes) and transcurrido + margen < HORAS_MAXIMAS:
        en_curso.append(lanzar(lotes[siguiente], libres.pop(0)))
        siguiente += 1
    terminado = None
    while terminado is None and en_curso:
        for t in en_curso:
            if t["proc"].poll() is not None:
                terminado = t
                break
        else:
            time.sleep(5)
    if terminado is None:
        break
    en_curso = [t for t in en_curso if t is not terminado]     # por identidad: comparar los lotes falla
    ok, dt = recoger(terminado)
    total_ok += ok
    segundos_por_lote.append(dt)
    print(f"Lote en GPU {terminado['gpu']}: {ok}/{len(terminado['lote'])} ok en {dt:.0f} s "
          f"({dt / len(terminado['lote']):.0f} s por serie) | {total_ok} máscaras | {(time.time() - inicio) / 3600:.2f} h")
    if len(segundos_por_lote) <= N_GPU and ok == 0:
        raise RuntimeError("El primer lote no produjo ninguna máscara: revisar el registro antes de seguir")

reg = pd.read_csv(CSV_REG, dtype=str) if CSV_REG.exists() else pd.DataFrame(columns=COLUMNAS_REG)
hechas_ahora = set(reg.loc[reg["estado"] == "ok", "id_serie"]) | hechas
faltan = lista[~lista["id_serie"].isin(hechas_ahora)].sort_values("orden")
print(f"\nEsta versión: {total_ok} máscaras | errores: {(reg['estado'] == 'error').sum()} | "
      f"huellas distintas: {(reg['huella_ok'] == 'False').sum()}")
print("Siguiente INICIO:", int(faltan["orden"].min()) if len(faltan) else "ninguno, la lista está completa")

GPUs: 2 | ya hechas en versiones anteriores: 0 | pendientes desde la posición 1: 2491
Lote en GPU 0: 20/20 ok en 517 s (26 s por serie) | 20 máscaras | 0.15 h
Lote en GPU 1: 20/20 ok en 517 s (26 s por serie) | 40 máscaras | 0.16 h
Lote en GPU 0: 20/20 ok en 515 s (26 s por serie) | 60 máscaras | 0.30 h
Lote en GPU 1: 20/20 ok en 513 s (26 s por serie) | 80 máscaras | 0.31 h
Lote en GPU 0: 20/20 ok en 489 s (24 s por serie) | 100 máscaras | 0.44 h
Lote en GPU 1: 20/20 ok en 513 s (26 s por serie) | 120 máscaras | 0.45 h
Lote en GPU 0: 20/20 ok en 516 s (26 s por serie) | 140 máscaras | 0.59 h
Lote en GPU 1: 20/20 ok en 512 s (26 s por serie) | 160 máscaras | 0.60 h
Lote en GPU 0: 20/20 ok en 514 s (26 s por serie) | 180 máscaras | 0.74 h
Lote en GPU 1: 20/20 ok en 510 s (25 s por serie) | 200 máscaras | 0.75 h
Lote en GPU 0: 20/20 ok en 495 s (25 s por serie) | 220 máscaras | 0.89 h
Lote en GPU 1: 20/20 ok en 509 s (25 s por serie) | 240 máscaras | 0.90 h
Lote en GPU 0: 20/20 ok en 513

In [5]:
# 5. Registrar el entorno (las máscaras ya están en el zip de salida)
from importlib.metadata import version as v
entorno = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "lista": LISTA, "inicio": INICIO,
           "python": sys.version.split()[0], "gpus": N_GPU,
           "gpu_nombre": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "totalspineseg": "20260807", "pesos_release": RELEASE_PESOS_TSS, "pesos_sha256": SHA256_PESOS_TSS,
           "tam_lote": TAM_LOTE, "simpleitk": v("SimpleITK"),
           "conversion08_sha256": sha256_archivo(RUTA_CONVERSION)}
(TRABAJO / f"entorno_{LISTA}_desde_{INICIO:05d}.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False))
print(json.dumps(entorno, indent=2, ensure_ascii=False))
with zipfile.ZipFile(ZIP_MASCARAS) as z:
    print(f"Máscaras en {ZIP_MASCARAS.name}: {len(z.namelist())} ({ZIP_MASCARAS.stat().st_size / 1e6:.1f} MB)")

{
  "fecha": "2026-10-30T12:37:46",
  "lista": "A",
  "inicio": 1,
  "python": "3.13.15",
  "gpus": 2,
  "gpu_nombre": "Tesla T4",
  "totalspineseg": "20260807",
  "pesos_release": "r20260730",
  "pesos_sha256": {
    "Dataset101_TotalSpineSeg_step1_r20260730.zip": "781f81e8c94feccd3cef17d819ba5c293b506e9b1f2b106fc5e31b9dd85887b0",
    "Dataset102_TotalSpineSeg_step2_r20260730.zip": "494c640ab8cb25475bd0fc4338c0931d37079af9dd19403879335c3908c06b46"
  },
  "tam_lote": 20,
  "simpleitk": "2.5.6",
  "conversion08_sha256": "404d686e9ab21673c80be9f8fd96d8b56f33a0999d51db041520bcf7a30a107f"
}
Máscaras en mascaras_A_desde_00001.zip: 2206 (51.2 MB)
